# Session 2 — How do ENSO and the IOD affect floods in Bangladesh?

**Workshop: ENSO, the IOD and floods in Bangladesh** · Session 2 of 2 · 60 minutes

Bangladesh sits at the outlet of three huge rivers: the **Brahmaputra** (Jamuna), the **Ganges** (Padma) and the **Meghna**. More than 90% of the land that drains through Bangladesh lies *outside* the country, in India, Nepal, Bhutan and China (Tibet), so most of the flood water falls as rain upstream. So to understand floods, we look at monsoon rain over the whole region and at river flow where the rivers enter Bangladesh.

In this session you will build a yearly flood indicator from river discharge, map how monsoon rainfall responds to ENSO and the IOD, test whether *combinations* of the two matter, and train two machine-learning models to predict flood years, while checking honestly whether they beat a simple baseline.

**How to use this notebook:** run each cell in order with **Shift + Enter**. Read the *What to look for* boxes and discuss. Cells marked **🔧 Try this** have settings you can change and re-run.

| Time | Part |
|---|---|
| 0–8 min | Context and setup |
| 8–20 min | Part 1 — A flood indicator from river discharge |
| 20–35 min | Part 2 — How does monsoon rainfall respond to ENSO and the IOD? |
| 35–45 min | Part 3 — Do the combinations matter? |
| 45–55 min | Part 4 — Can machine learning predict flood years? |
| 55–60 min | Discussion |

**Data used:** GloFAS v4 river discharge reanalysis (Copernicus), ERA5 monthly precipitation (Copernicus), NOAA ERSSTv5 sea surface temperature.

## Setup — load the workshop data

This cell finds the workshop data. In Colab it first copies the workshop repository from GitHub, which takes up to a minute the first time; after that it is instant.

In [ ]:
import os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats

# Find the workshop data. In Colab we copy (clone) the workshop repository first;
# when running from a downloaded copy of the repository, its data folder is used directly.
REPO_URL = "https://github.com/pazolka/ai-climate-workshop-dhaka.git"
DATA_DIR = None
for candidate in ["../data", "data", "ai-climate-workshop-dhaka/data"]:
    if os.path.exists(os.path.join(candidate, "gridded", "sst_indopacific.npz")):
        DATA_DIR = candidate
        break
if DATA_DIR is None:
    print("Copying the workshop repository (about 100 MB, takes up to a minute)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    DATA_DIR = "ai-climate-workshop-dhaka/data"

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty land cells give harmless warnings
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Workshop data found in:", DATA_DIR)

In [ ]:
# Helper functions (you don't need to read these — just run the cell)

def season_months(season):
    """'JJAS' -> [6, 7, 8, 9];  'DJF' -> [12, 1, 2]"""
    s = "JFMAMJJASOND" * 2
    i = s.find(season)
    if i < 0:
        raise ValueError(f"Unknown season {season}")
    return [(i + k) % 12 + 1 for k in range(len(season))]


def seasonal_mean(series, season):
    """Mean of a monthly pandas Series over a season, one value per year.
    For seasons crossing the new year (e.g. DJF) the year of January is used."""
    months = season_months(season)
    df = pd.DataFrame({"v": series.values, "m": series.index.month, "y": series.index.year})
    if months[-1] < months[0]:
        df.loc[df.m >= months[0], "y"] += 1
    sel = df[df.m.isin(months)]
    g = sel.groupby("y")["v"]
    return g.mean()[g.count() == len(months)]


def seasonal_maps(field, times, season):
    """Seasonal mean of a (time, lat, lon) array -> years, (year, lat, lon) array."""
    months = season_months(season)
    yrs = times.year.to_numpy().copy()
    if months[-1] < months[0]:
        yrs[times.month >= months[0]] += 1
    keep = np.isin(times.month, months)
    years, maps = [], []
    for y in np.unique(yrs[keep]):
        idx = keep & (yrs == y)
        if idx.sum() == len(months):
            years.append(y)
            maps.append(np.nanmean(field[idx], axis=0))
    return np.array(years), np.array(maps)


def box_mean(field, lat, lon, lat_s, lat_n, lon_w, lon_e):
    """Area-weighted average of a (time, lat, lon) array over a lat/lon box."""
    la = (lat >= lat_s) & (lat <= lat_n)
    lo = (lon >= lon_w) & (lon <= lon_e)
    sub = field[:, la][:, :, lo]
    w = np.cos(np.deg2rad(lat[la]))[:, None] * np.ones(lo.sum())[None, :]
    valid = ~np.isnan(sub)
    return np.nansum(sub * w, axis=(1, 2)) / np.sum(w * valid, axis=(1, 2))


def anomalies(field, times, base=(1991, 2020), detrend=True):
    """Remove the monthly climatology (and optionally a linear trend) at every grid point."""
    anom = np.full_like(field, np.nan)
    in_base = (times.year >= base[0]) & (times.year <= base[1])
    for m in range(1, 13):
        sel = times.month == m
        anom[sel] = field[sel] - np.nanmean(field[sel & in_base], axis=0)
    if detrend:
        t = np.arange(len(times), dtype=float)
        flat = anom.reshape(len(t), -1)
        ok = ~np.isnan(flat).any(axis=0)
        coef = np.polyfit(t, flat[:, ok], 1)
        flat[:, ok] -= np.outer(t, coef[0]) + coef[1]
        anom = flat.reshape(anom.shape)
    return anom


NINO34_BOX = dict(lat_s=-5, lat_n=5, lon_w=190, lon_e=240)
DMI_WEST = dict(lat_s=-10, lat_n=10, lon_w=50, lon_e=70)
DMI_EAST = dict(lat_s=-10, lat_n=0, lon_w=90, lon_e=110)


def plot_map(ax, field, lat, lon, title="", vmax=1.5, cmap="RdBu_r", boxes=True):
    ax.set_facecolor("0.85")  # land shows as grey
    pm = ax.pcolormesh(lon, lat, field, cmap=cmap, vmin=-vmax, vmax=vmax, shading="auto")
    if boxes:
        for b, c in [(NINO34_BOX, "k"), (DMI_WEST, "purple"), (DMI_EAST, "purple")]:
            ax.add_patch(Rectangle((b["lon_w"], b["lat_s"]), b["lon_e"] - b["lon_w"],
                                   b["lat_n"] - b["lat_s"], fill=False, ec=c, lw=1.2))
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=10, loc="left")
    ax.grid(False)
    ax.set_xticks(range(40, 300, 40))
    ax.set_xticklabels([f"{x}°E" if x <= 180 else f"{360 - x}°W" for x in range(40, 300, 40)], fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    return pm

print("Helpers ready.")

In [ ]:
# Rebuild the Niño 3.4 and DMI indices exactly as in Session 1
d = np.load(f"{DATA_DIR}/gridded/sst_indopacific.npz")
sst, lat, lon = d["sst"], d["lat"], d["lon"]
times = pd.DatetimeIndex(pd.to_datetime(pd.Series(d["time"]).astype(str) + "-01"))
sst_anom = anomalies(sst, times, detrend=True)
nino34 = pd.Series(box_mean(sst_anom, lat, lon, **NINO34_BOX), index=times)
dmi = pd.Series(box_mean(sst_anom, lat, lon, **DMI_WEST) - box_mean(sst_anom, lat, lon, **DMI_EAST), index=times)


def classify(x, thr, pos, neg):
    z = (x - x.mean()) / x.std()
    return pd.Series(np.where(z > thr, pos, np.where(z < -thr, neg, "Neutral")), index=x.index)

print(f"Indices ready: {times[0]:%b %Y} – {times[-1]:%b %Y}")

## Part 1 — A flood indicator from river discharge (≈12 min)

We use **GloFAS**, a global river model driven by weather reanalysis, which gives daily discharge (m³/s) everywhere, including where gauge records are hard to obtain. We take two key points where the rivers enter the delta:

- **Bahadurabad** on the Brahmaputra (Jamuna) — the largest flow, fed by the eastern Himalaya and Tibet.
- **Hardinge Bridge** on the Ganges (Padma) — fed by northern India and Nepal.

Note: GloFAS is a model, not observations. It captures the timing and relative size of floods well, but its absolute numbers can differ from Bangladesh's own gauges.

In [ ]:
q = pd.read_csv(f"{DATA_DIR}/point_observations/glofas_discharge.csv", parse_dates=["date"], index_col="date")
STATIONS = {"bahadurabad": "Brahmaputra at Bahadurabad", "hardinge_bridge": "Ganges at Hardinge Bridge"}
print(f"Daily discharge {q.index[0]:%Y-%m-%d} to {q.index[-1]:%Y-%m-%d}")

YEAR_TO_SHOW = 1998   # 🔧 Try this: 1988, 2004, 2007, 2017, 2020 or a dry year like 1992

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
for ax, (col, title) in zip(axes, STATIONS.items()):
    df = pd.DataFrame({"q": q[col] / 1000, "doy": q.index.dayofyear, "y": q.index.year})
    clim = df.groupby("doy")["q"].quantile([0.1, 0.5, 0.9]).unstack()
    ax.fill_between(clim.index, clim[0.1], clim[0.9], color="0.8", label="10–90% range, all years")
    ax.plot(clim.index, clim[0.5], color="0.3", label="median")
    yr = df[df.y == YEAR_TO_SHOW]
    ax.plot(yr["doy"], yr["q"], color="tab:red", lw=1.5, label=str(YEAR_TO_SHOW))
    ax.set_xticks([1, 91, 182, 274]); ax.set_xticklabels(["Jan", "Apr", "Jul", "Oct"])
    ax.set_title(title, loc="left"); ax.set_ylabel("discharge (thousand m³/s)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

> **What to look for:** when does each river start to rise, peak and fall? Which river peaks first? How was 1998 different from a normal year, in height and in how long the water stayed high? What would happen in the delta if both rivers peaked at the same time?

Now we turn each year into one number: the **highest daily discharge during June–October** (the annual monsoon peak). A year counts as a **flood year** if its peak is in the top third of all years.

In [ ]:
FLOOD_PERCENTILE = 67   # 🔧 Try this: 75 or 80 for "big floods only"

# Major floods reported in the literature, for comparison (not used in the calculation)
KNOWN_FLOODS = [1987, 1988, 1998, 2004, 2007, 2017, 2020]

monsoon = q[q.index.month.isin([6, 7, 8, 9, 10])]
counts = monsoon.groupby(monsoon.index.year).count().min(axis=1)
peaks = monsoon.groupby(monsoon.index.year).max()[counts >= 150]  # complete seasons only
flood = peaks > peaks.quantile(FLOOD_PERCENTILE / 100)
peaks_z = (peaks - peaks.mean()) / peaks.std()
flood["either"] = flood.any(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
for ax, (col, title) in zip(axes, STATIONS.items()):
    colors = np.where(flood[col], "tab:blue", "0.7")
    ax.bar(peaks.index, peaks[col] / 1000, color=colors)
    ax.axhline(peaks[col].quantile(FLOOD_PERCENTILE / 100) / 1000, color="k", ls="--", lw=0.8)
    for y in KNOWN_FLOODS:
        if y in peaks.index:
            ax.plot(y, peaks.loc[y, col] / 1000 * 1.04, "v", color="tab:red", ms=6)
    ax.set_ylabel("peak (thousand m³/s)")
    ax.set_title(f"{title} — blue: flood year (top {100 - FLOOD_PERCENTILE}%), ▼ known major flood", loc="left")
    ax.set_ylim(peaks[col].min() / 1000 * 0.8, peaks[col].max() / 1000 * 1.12)
plt.tight_layout(); plt.show()

print("Flood years (either river):", list(flood.index[flood["either"]]))
print("Correlation between the two rivers' peaks: r =", round(peaks.corr().iloc[0, 1], 2))

> **What to look for**
> - Do the red ▼ known floods line up with the blue bars? If some do not, why might that be? (Think about how long the water stays high, both rivers peaking together, local rain, or the Meghna.)
> - Look at the printed correlation between the two rivers' peaks. Are big Brahmaputra years also big Ganges years?
>
> **Discuss:** Is "peak discharge" the best flood indicator? What else could we use? (Days above danger level, flood volume, flooded area from satellites…)

## Part 2 — How does monsoon rainfall respond to ENSO and the IOD? (≈15 min)

We correlate monsoon (June–September) rainfall at every grid point with Niño 3.4 and with the DMI, both averaged over the same season.

Then we compute **partial correlations**: the link with ENSO *after removing* the part shared with the IOD, and vice versa. Because ENSO and the IOD are correlated (Session 1), a plain correlation map for one index partly shows the other.

Dotted areas are **statistically significant** at the 95% level. With ~47 years, any correlation smaller than about ±0.3 could easily be chance.

In [ ]:
SEASON = "JJAS"   # 🔧 Try this: "JJA", "JAS" or "SON" (post-monsoon)

p = np.load(f"{DATA_DIR}/gridded/era5_precip_gbm.npz")
precip, plat, plon, lsm = p["precip"], p["lat"], p["lon"], p["lsm"]
ptimes = pd.DatetimeIndex(pd.to_datetime(pd.Series(p["time"]).astype(str) + "-01"))

yrs_p, rain_maps = seasonal_maps(precip, ptimes, SEASON)
n_s, d_s = seasonal_mean(nino34, SEASON), seasonal_mean(dmi, SEASON)
common = sorted(set(yrs_p) & set(n_s.index) & set(d_s.index))
R = rain_maps[np.isin(yrs_p, common)]
xn, xd = n_s.loc[common].values, d_s.loc[common].values
n_years = len(common)


def corr_map(x, F):
    xz = (x - x.mean()) / x.std()
    Fz = (F - F.mean(axis=0)) / F.std(axis=0)
    return np.mean(xz[:, None, None] * Fz, axis=0)


def partial(r_xy, r_xz, r_yz):
    return (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz ** 2) * (1 - r_yz ** 2))


r_n, r_d = corr_map(xn, R), corr_map(xd, R)
r_nd = np.corrcoef(xn, xd)[0, 1]
pr_n, pr_d = partial(r_n, r_d, r_nd), partial(r_d, r_n, r_nd)
t = stats.t.ppf(0.975, n_years - 2)
r_crit = t / np.sqrt(n_years - 2 + t ** 2)

STATION_XY = {"Bahadurabad": (89.67, 25.18), "Hardinge Br.": (89.03, 24.07)}


def plot_region(ax, field, title, vmax=0.6):
    pm = ax.pcolormesh(plon, plat, field, cmap="BrBG", vmin=-vmax, vmax=vmax, shading="auto")
    ax.contourf(plon, plat, (np.abs(field) > r_crit).astype(float), levels=[0.5, 1.5],
                hatches=[".."], colors="none")
    ax.contour(plon, plat, lsm, levels=[0.5], colors="k", linewidths=0.8)
    for name, (x, y) in STATION_XY.items():
        ax.plot(x, y, "o", color="tab:red", ms=5, mec="k")
    ax.set_aspect("equal"); ax.grid(False)
    ax.set_title(title, fontsize=10, loc="left")
    return pm


fig, axes = plt.subplots(2, 2, figsize=(12, 6.2), layout="constrained")
pm = plot_region(axes[0, 0], r_n, f"Correlation with Niño 3.4 ({SEASON})")
plot_region(axes[0, 1], r_d, f"Correlation with DMI ({SEASON})")
plot_region(axes[1, 0], pr_n, "Partial: Niño 3.4, IOD removed")
plot_region(axes[1, 1], pr_d, "Partial: DMI, ENSO removed")
fig.colorbar(pm, ax=axes, shrink=0.7, label="correlation (green = wetter when index is positive)")
fig.suptitle(f"{SEASON} rainfall vs climate indices, {common[0]}–{common[-1]} (n={n_years}). "
             f"Dots: |r| > {r_crit:.2f} (95%). Red points: discharge stations.", fontsize=10)
plt.show()
print(f"Correlation between Niño 3.4 and DMI in {SEASON}: {r_nd:.2f}")

> **What to look for**
> - **Niño 3.4 (top left):** where is monsoon rain drier (brown) or wetter (green) in El Niño years? What happens over the upper Ganges basin (Nepal, northern India), and over Bangladesh itself?
> - **DMI (top right):** where does a positive IOD bring more rain? Is it the same region where El Niño brings less?
> - **Partial maps (bottom):** do the patterns get stronger or weaker once the other index is removed? If they get stronger, what does that tell you about how the two modes combine?
> - **Dots:** how much of each map is significant? Be careful about interpreting colours without dots.
>
> **Discuss:** Look at where the Ganges basin (northern India, Nepal) and the Brahmaputra basin (north-east India, Bhutan, Tibet) lie on these maps. Do you expect the two rivers to respond in the same way? The next cell checks.

### From rain maps to the two rivers

The maps show *where* rain responds, but the two rivers drain different areas. Here we average the rainfall over each river's catchment and test both links in the chain:

1. **Does this rain drive the river?** Correlate each area's rainfall with each river's yearly flood peak from Part 1.
2. **Does this rain respond to the oceans?** Correlate the same rainfall with Niño 3.4 and the DMI, using partial correlations to separate the two.

The boxes are rough stand-ins for the real basins, drawn on the map below.

In [ ]:
# Average the rainfall over each river's catchment, then test both links in the chain:
#   (1) does this rain drive the river's flood peak?   (2) does this rain respond to ENSO and the IOD?
# The boxes are rough stand-ins for the real basins (land points only).
BASIN_BOXES = {
    "Ganges: northern India & Nepal": dict(lat_s=24.0, lat_n=30.0, lon_w=77.0, lon_e=88.0, color="tab:orange"),
    "Brahmaputra: Assam, Bhutan & Arunachal": dict(lat_s=25.5, lat_n=29.0, lon_w=89.0, lon_e=96.0, color="tab:green"),
}

basin_rain = {}
for name, b in BASIN_BOXES.items():
    la = (plat >= b["lat_s"]) & (plat <= b["lat_n"])
    lo = (plon >= b["lon_w"]) & (plon <= b["lon_e"])
    land = lsm[la][:, lo] > 0.5
    box = np.where(land, precip[:, la][:, :, lo], np.nan)
    basin_rain[name] = seasonal_mean(pd.Series(np.nanmean(box, axis=(1, 2)), index=ptimes), SEASON)
basin_rain = pd.DataFrame(basin_rain)

nb_, db_ = seasonal_mean(nino34, SEASON), seasonal_mean(dmi, SEASON)
yrs_b = sorted(set(basin_rain.index) & set(nb_.index) & set(db_.index) & set(peaks.index))
r_nd_b = nb_.loc[yrs_b].corr(db_.loc[yrs_b])
rows = []
for name in basin_rain:
    x = basin_rain.loc[yrs_b, name]
    rn_b, rd_b = x.corr(nb_.loc[yrs_b]), x.corr(db_.loc[yrs_b])
    rows.append({"rainfall area": name,
                 "r with Ganges peak": x.corr(peaks.loc[yrs_b, "hardinge_bridge"]),
                 "r with Brahmaputra peak": x.corr(peaks.loc[yrs_b, "bahadurabad"]),
                 "r with Niño 3.4": rn_b, "r with DMI": rd_b,
                 "Niño 3.4, IOD removed": partial(rn_b, rd_b, r_nd_b),
                 "DMI, ENSO removed": partial(rd_b, rn_b, r_nd_b)})
basin_table = pd.DataFrame(rows).set_index("rainfall area")
t_b = stats.t.ppf(0.975, len(yrs_b) - 2)
r_crit_b = t_b / np.sqrt(len(yrs_b) - 2 + t_b ** 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
clim_rain = np.nanmean(seasonal_maps(precip, ptimes, SEASON)[1], axis=0)
pm = ax.pcolormesh(plon, plat, clim_rain, cmap="Blues", vmax=np.nanpercentile(clim_rain, 97), shading="auto")
ax.contour(plon, plat, lsm, levels=[0.5], colors="k", linewidths=0.8)
for name, b in BASIN_BOXES.items():
    ax.add_patch(Rectangle((b["lon_w"], b["lat_s"]), b["lon_e"] - b["lon_w"], b["lat_n"] - b["lat_s"],
                           fill=False, ec=b["color"], lw=2.5))
    ax.text(b["lon_w"] + 0.3, b["lat_n"] - 0.8, name.split(":")[0], color=b["color"], fontsize=10, fontweight="bold")
for nm, (x, y) in STATION_XY.items():
    ax.plot(x, y, "o", color="tab:red", ms=5, mec="k")
ax.set_aspect("equal"); ax.grid(False)
ax.set_title(f"Mean {SEASON} rainfall (mm/day) and the two rainfall areas", loc="left", fontsize=10)
plt.colorbar(pm, ax=ax, shrink=0.75)

ax = axes[1]
cols = ["r with Ganges peak", "r with Brahmaputra peak", "Niño 3.4, IOD removed", "DMI, ENSO removed"]
xpos = np.arange(len(cols))
for k, (name, b) in enumerate(BASIN_BOXES.items()):
    ax.bar(xpos + (k - 0.5) * 0.38, basin_table.loc[name, cols], 0.38, color=b["color"], label=name)
for v in (r_crit_b, -r_crit_b):
    ax.axhline(v, color="k", ls="--", lw=0.8)
ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(xpos); ax.set_xticklabels(["drives Ganges\npeak?", "drives Brahmaputra\npeak?", "responds to\nENSO?", "responds to\nIOD?"])
ax.set_ylabel("correlation")
ax.set_title(f"Both links of the chain, {yrs_b[0]}–{yrs_b[-1]} (dashed: significant at 95%)", loc="left", fontsize=10)
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

print(f"{SEASON} rainfall averaged over each area (land only), {len(yrs_b)} years. |r| > {r_crit_b:.2f} is significant at 95%.")
basin_table.round(2)

> **What to look for**
> - **Which rain drives which river?** Compare the first two groups of bars. Does each river follow the rain in its own catchment?
> - **Which rain responds to the oceans?** Compare the last two groups. Which area's rain crosses the significance lines?
> - **Putting the chain together:** for the ocean signal to reach a river, both links must be strong. Is that true for the Ganges? For the Brahmaputra? What do you expect Part 3 to show?
> - **Caveat:** the boxes are approximate. The real Brahmaputra basin also extends far into Tibet, which this data covers only partly.
>
> **Try this:** change `SEASON` in the Part 2 cell and re-run both cells. Does the picture hold in June–August or July–September?

## Part 3 — Do the combinations matter? (≈10 min)

Now we go straight to river flow. First, plain and partial correlations between each index and the monsoon peak at each station. Then a picture of every year on the ENSO–IOD plane.

In [ ]:
yrs = sorted(set(peaks.index) & set(n_s.index) & set(d_s.index))
tab = pd.DataFrame({"nino34": n_s.loc[yrs], "dmi": d_s.loc[yrs]})
for col in STATIONS:
    tab[col] = peaks_z.loc[yrs, col]

rows = []
for col, title in STATIONS.items():
    r_qn = tab[col].corr(tab["nino34"]); r_qd = tab[col].corr(tab["dmi"]); r_nd2 = tab["nino34"].corr(tab["dmi"])
    rows.append({"river": title,
                 "r with Niño 3.4": r_qn, "r with DMI": r_qd,
                 "partial r, Niño 3.4 (IOD removed)": partial(r_qn, r_qd, r_nd2),
                 "partial r, DMI (ENSO removed)": partial(r_qd, r_qn, r_nd2)})
t2 = stats.t.ppf(0.975, len(yrs) - 2)
print(f"{SEASON} indices vs June–October peak discharge, {yrs[0]}–{yrs[-1]}. "
      f"|r| > {t2 / np.sqrt(len(yrs) - 2 + t2 ** 2):.2f} is significant at 95%.")
pd.DataFrame(rows).set_index("river").round(2)

In [ ]:
TARGET = "bahadurabad"   # 🔧 Try this: "hardinge_bridge"


fig, ax = plt.subplots(figsize=(8, 6.5))
v = tab[TARGET]
sc = ax.scatter(tab["nino34"], tab["dmi"], c=v, cmap="RdYlBu_r", vmin=-2, vmax=2, s=90, edgecolor="k")
for y in yrs:
    ring = y in KNOWN_FLOODS
    ax.annotate(str(y), (tab.loc[y, "nino34"], tab.loc[y, "dmi"]), fontsize=7,
                xytext=(4, 4), textcoords="offset points", fontweight="bold" if ring else "normal")
    if ring:
        ax.plot(tab.loc[y, "nino34"], tab.loc[y, "dmi"], "o", ms=14, mfc="none", mec="k", mew=1.5)
ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel(f"Niño 3.4, {SEASON} (°C)   ← La Niña | El Niño →")
ax.set_ylabel(f"DMI, {SEASON} (°C)   ← negative IOD | positive IOD →")
ax.set_title(f"{STATIONS[TARGET]}: monsoon peak (colour, in standard deviations). Rings: known major floods", loc="left", fontsize=10)
plt.colorbar(sc, ax=ax, label="peak discharge anomaly (SD)")
plt.show()

In [ ]:
# Average flood peak in each ENSO–IOD combination (same ±0.75 SD rule as Session 1)
enso_c = classify(tab["nino34"], 0.75, "El Niño", "La Niña")
iod_c = classify(tab["dmi"], 0.75, "IOD+", "IOD-")
rows_o, cols_o = ["El Niño", "Neutral", "La Niña"], ["IOD-", "Neutral", "IOD+"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (col, title) in zip(axes, STATIONS.items()):
    mean = tab[col].groupby([enso_c, iod_c]).mean().unstack().reindex(index=rows_o, columns=cols_o)
    n = tab[col].groupby([enso_c, iod_c]).count().unstack().reindex(index=rows_o, columns=cols_o).fillna(0)
    im = ax.imshow(mean.values, cmap="RdYlBu_r", vmin=-1.2, vmax=1.2)
    for i in range(3):
        for j in range(3):
            val = mean.values[i, j]
            ax.text(j, i, "–" if np.isnan(val) else f"{val:+.2f}\nn={int(n.values[i, j])}",
                    ha="center", va="center", fontsize=9,
                    color="white" if abs(np.nan_to_num(val)) > 0.8 else "black")
    ax.set_xticks(range(3)); ax.set_xticklabels(cols_o)
    ax.set_yticks(range(3)); ax.set_yticklabels(rows_o)
    ax.grid(False); ax.set_title(title, loc="left", fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="mean peak anomaly (SD)")
fig.suptitle(f"Mean monsoon peak discharge by {SEASON} ENSO–IOD category", fontsize=11)
plt.show()

> **What to look for**
> - In the scatter plot, are the red (high-flow) dots clustered in one corner, or spread out? A clear corner means the *combination* matters.
> - In the grid, compare cells with similar ENSO but different IOD. Does adding the IOD change the average?
> - **Look at n.** A cell with n = 2 or 3 tells you very little. An average of +0.8 from three years could easily be luck.
>
> **Discuss:** Several big floods (1988, 1998, 2007) happened when an El Niño had just ended and La Niña was developing. Our seasonal snapshot can miss that *transition*. How could we capture it? (Hint: compare the previous winter's Niño 3.4 with the current summer's.)

## Part 4 — Can machine learning predict flood years? (≈10 min)

We now train models to predict whether a year is a **flood year** (blue bars in Part 1) from Niño 3.4 and the DMI.

We compare three forecasts:
1. **Climatology** (the baseline): always predict the long-term flood frequency (about 33%). Any useful model must beat this.
2. **Logistic regression:** a simple statistical model. We add an ENSO × IOD *interaction* term so it can capture combinations.
3. **Random forest:** a machine-learning model made of many decision trees, which can find non-linear patterns and interactions on its own.

**Leave-one-year-out cross-validation:** for each year, we train on all *other* years and predict the left-out year. This is the honest way to test with so little data. A model is never tested on a year it has seen.

**Two settings to explore:**
- `FEATURE_SEASON = "JJAS"` uses indices from the *same* season as the flood. This is a **diagnostic** test (does the climate state explain floods?), but not a real forecast.
- `FEATURE_SEASON = "MAM"` or `"AMJ"` uses spring indices, available *before* the monsoon starts. This is a **real forecast** test.

We score the forecasts with the **Brier skill score (BSS)**: 0 = no better than climatology, positive = better, negative = worse. We also show the **ROC AUC**: 0.5 = no skill, 1 = perfect ranking.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss, roc_auc_score

FEATURE_SEASON = "JJAS"     # 🔧 Try this: "MAM" or "AMJ" (a real pre-season forecast)
TARGET_ML = "bahadurabad"   # 🔧 Try this: "hardinge_bridge" or "either"


feat = pd.DataFrame({"nino34": seasonal_mean(nino34, FEATURE_SEASON),
                     "dmi": seasonal_mean(dmi, FEATURE_SEASON)})
data = feat.join(flood[TARGET_ML].rename("flood"), how="inner").dropna()
X = data[["nino34", "dmi"]].values
y = data["flood"].astype(int).values
years_ml = data.index.values


def with_interaction(A):
    return np.column_stack([A, A[:, 0] * A[:, 1]])


def make_logistic():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0))


def make_forest():
    return RandomForestClassifier(n_estimators=300, max_depth=3, min_samples_leaf=3, random_state=0)


prob = {"Climatology": np.zeros(len(y)), "Logistic regression": np.zeros(len(y)), "Random forest": np.zeros(len(y))}
for i in range(len(y)):
    tr = np.arange(len(y)) != i
    prob["Climatology"][i] = y[tr].mean()
    prob["Logistic regression"][i] = make_logistic().fit(with_interaction(X[tr]), y[tr]).predict_proba(with_interaction(X[i:i + 1]))[0, 1]
    prob["Random forest"][i] = make_forest().fit(X[tr], y[tr]).predict_proba(X[i:i + 1])[0, 1]

bs_clim = brier_score_loss(y, prob["Climatology"])
scores = pd.DataFrame({name: {"Brier score": brier_score_loss(y, pr),
                              "Brier skill score": 1 - brier_score_loss(y, pr) / bs_clim,
                              "ROC AUC": roc_auc_score(y, pr) if name != "Climatology" else 0.5}
                       for name, pr in prob.items()}).T.round(3)
print(f"Target: {TARGET_ML}, {y.sum()} flood years out of {len(y)}; predictors: {FEATURE_SEASON} Niño 3.4 and DMI\n")
print(scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 3]})
axes[0].bar(scores.index, scores["Brier skill score"], color=["0.6", "tab:blue", "tab:green"])
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_title("Brier skill score (>0 beats climatology)", loc="left", fontsize=10)
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(years_ml, y, color="tab:blue", alpha=0.2, width=1.0, label="flood year")
axes[1].plot(years_ml, prob["Logistic regression"], "o-", color="tab:blue", ms=3, label="logistic")
axes[1].plot(years_ml, prob["Random forest"], "s-", color="tab:green", ms=3, label="random forest")
axes[1].axhline(y.mean(), color="0.4", ls="--", lw=0.8, label="climatology")
axes[1].set_ylim(0, 1); axes[1].set_ylabel("predicted flood probability")
axes[1].set_title("Leave-one-year-out predictions (each year predicted by a model that never saw it)", loc="left", fontsize=10)
axes[1].legend(fontsize=8, ncol=4)
plt.tight_layout(); plt.show()

### What did the models learn?

With only two predictors we can draw each model's predicted flood probability for every possible combination of Niño 3.4 and DMI. The dots are the real years (orange = flood year). Then we use **SHAP**, a popular explainable-AI method, to split each random forest prediction into a contribution from ENSO and a contribution from the IOD.

In [ ]:
logi_all = make_logistic().fit(with_interaction(X), y)
rf_all = make_forest().fit(X, y)

gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 120),
                     np.linspace(X[:, 1].min() - 0.15, X[:, 1].max() + 0.15, 120))
grid = np.column_stack([gx.ravel(), gy.ravel()])
surfaces = {"Logistic regression (with interaction)": logi_all.predict_proba(with_interaction(grid))[:, 1],
            "Random forest": rf_all.predict_proba(grid)[:, 1]}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (name, z) in zip(axes, surfaces.items()):
    cf = ax.contourf(gx, gy, z.reshape(gx.shape), levels=np.linspace(0, 1, 11), cmap="PuBu", alpha=0.85)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="white", edgecolor="k", s=35, label="normal year")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:orange", edgecolor="k", s=50, label="flood year")
    for yr, (a, b) in zip(years_ml, X):
        ax.annotate(str(yr)[2:], (a, b), fontsize=6.5, xytext=(3, 3), textcoords="offset points")
    ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
    ax.set_xlabel(f"Niño 3.4, {FEATURE_SEASON}"); ax.set_title(name, loc="left", fontsize=10); ax.grid(False)
axes[0].set_ylabel(f"DMI, {FEATURE_SEASON}"); axes[0].legend(fontsize=8, loc="lower right")
fig.colorbar(cf, ax=axes, shrink=0.85, label="predicted flood probability")
plt.show()

In [ ]:
# SHAP explanation of the random forest (falls back to permutation importance if SHAP is unavailable)
try:
    import shap
except ImportError:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True, timeout=180, capture_output=True)
        import shap
    except Exception:
        shap = None

if shap is not None:
    sv = shap.TreeExplainer(rf_all).shap_values(X)
    sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if np.ndim(sv) == 3 else sv)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].bar(["Niño 3.4", "DMI"], np.abs(sv).mean(axis=0), color=["tab:red", "tab:purple"])
    axes[0].set_title("Mean |SHAP|: overall importance", loc="left", fontsize=10)
    for ax, k, other, nm, onm in [(axes[1], 0, 1, "Niño 3.4", "DMI"), (axes[2], 1, 0, "DMI", "Niño 3.4")]:
        s = ax.scatter(X[:, k], sv[:, k], c=X[:, other], cmap="coolwarm", edgecolor="k", s=35)
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlabel(f"{nm} ({FEATURE_SEASON})"); ax.set_ylabel(f"SHAP value of {nm}")
        ax.set_title(f"Effect of {nm} (colour = {onm})", loc="left", fontsize=10)
        plt.colorbar(s, ax=ax, label=onm)
    plt.tight_layout(); plt.show()
else:
    from sklearn.inspection import permutation_importance
    imp = permutation_importance(rf_all, X, y, n_repeats=50, random_state=0, scoring="neg_brier_score")
    plt.figure(figsize=(5, 3))
    plt.bar(["Niño 3.4", "DMI"], imp.importances_mean, yerr=imp.importances_std, color=["tab:red", "tab:purple"])
    plt.title("Permutation importance (SHAP not available)", loc="left", fontsize=10)
    plt.show()

> **What to look for**
> - **Skill scores:** does either model have a positive Brier skill score, beating climatology? Compare the same-season (JJAS) and spring (MAM or AMJ) settings, and the two rivers. Which set-ups, if any, show skill?
> - **Logistic vs random forest:** is the more complex model better? What does the random forest's blocky probability map suggest about how it learns from ~47 years?
> - **Probability maps:** is the high-probability area a single corner (one index matters), a band, or a diagonal (both indices matter together)?
> - **SHAP:** a SHAP value above zero means that index pushed the prediction *towards* a flood year. Does the effect of ENSO depend on the IOD colour (an interaction)?
> - **Before you trust a score:** how many set-ups did your group try? Run the shuffle test below on the one that looked best.

### How lucky could we get? (optional, ≈1 min to run)

With few years, a model can score well just by chance. Here we shuffle the flood labels randomly many times, so there is *no* real link with the climate indices, and record the skill scores the logistic model achieves anyway. If our real score is not clearly better than most shuffled scores, we have not shown real skill.

In [ ]:
N_SHUFFLES = 200   # 🔧 increase for a smoother picture (slower)
rng = np.random.default_rng(0)


def loo_bss_logistic(Xf, yf):
    pr, pc = np.zeros(len(yf)), np.zeros(len(yf))
    for i in range(len(yf)):
        tr = np.arange(len(yf)) != i
        pc[i] = yf[tr].mean()
        if yf[tr].min() == yf[tr].max():
            pr[i] = pc[i]
            continue
        pr[i] = make_logistic().fit(with_interaction(Xf[tr]), yf[tr]).predict_proba(with_interaction(Xf[i:i + 1]))[0, 1]
    return 1 - brier_score_loss(yf, pr) / brier_score_loss(yf, pc)


real = scores.loc["Logistic regression", "Brier skill score"]
fake = np.array([loo_bss_logistic(X, rng.permutation(y)) for _ in range(N_SHUFFLES)])
p_value = (np.sum(fake >= real) + 1) / (N_SHUFFLES + 1)

plt.figure(figsize=(7, 3.2))
plt.hist(fake, bins=25, color="0.7", label="shuffled labels (no real link)")
plt.axvline(real, color="tab:blue", lw=2, label=f"real data: BSS = {real:.2f}")
plt.xlabel("Brier skill score"); plt.legend(fontsize=8)
plt.title(f"Chance of doing this well by luck: p ≈ {p_value:.2f}", loc="left", fontsize=10)
plt.show()

## Discussion (≈5 min)

**Questions to discuss**
1. **Rainfall:** do ENSO and the IOD leave a fingerprint on monsoon rain? Is it the same over the Ganges and the Brahmaputra catchments?
2. **Combinations:** can the two modes offset each other? Is knowing both more useful than knowing one?
3. **Floods:** how strong is the link between the ocean states and the flood peaks? What else drives floods that our ocean indices cannot see?
4. **Machine learning:** did a complex model beat a simple one? Did any model beat climatology, and did it pass the shuffle test?
5. Would you issue a seasonal flood outlook for Bangladesh based on your results? What would you tell a disaster manager about its reliability?
6. What extra data could help? (Longer records, other indices, climate-model simulations, sub-seasonal signals.)

**Where to go next** (optional take-home ideas)
- **Transition years:** add the previous winter's Niño 3.4 as a third predictor to test the "decaying El Niño" idea.
- **Sub-seasonal timing:** look at the MJO / Boreal Summer Intraseasonal Oscillation phases before big flood peaks.
- **Deep learning with daily data:** train an LSTM to predict daily discharge from rainfall, where thousands of samples make deep learning appropriate.
- **Satellite flood mapping:** map flooded areas from Sentinel-1 radar for 2017, 2020 or 2024.
- **Impacts:** compare flood years with detrended Aman rice yield anomalies.

**Data credits:** NOAA ERSSTv5 (NOAA Physical Sciences Laboratory). ERA5: contains modified Copernicus Climate Change Service information (2026). GloFAS v4: contains modified Copernicus Emergency Management Service information (2026).